# A — Gradient Descent Regression

**Maths for Machine Learning · International MSc Finance & Data — Albert School · 2026**

Implement linear regression with **gradient descent from scratch**, tune learning rates, compare convergence, and benchmark against the **closed-form OLS** solution. Then freely explore the data using concepts from the course.

**Dataset:** [House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) (Kaggle).

**Concepts mobilised**

| Session | Topic | Application in this project |
|---|---|---|
| 1 — Linear Algebra | Vectors, matrices, eigenvalues, PCA | Data as $X \in \mathbb{R}^{n \times d}$, OLS via $(X^\top X)^{-1}$, Hessian spectrum, PCA |
| 2 — Calculus & Optimisation | Derivatives, gradient, Hessian | MSE gradient, GD update rule, learning rate bound |
| 3 — Probability | Distributions, normality | Log-transform, Q-Q plot, Shapiro-Wilk test |
| 4 — Statistics | Hypothesis testing | Welch's t-test on quality groups |
| 5 — Advanced Optimisation | Regularisation | Ridge ($L_2$) regression from scratch |

---
## 0 · Setup

In [ ]:
%pip install -q kagglehub pandas numpy matplotlib seaborn scipy

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from mpl_toolkits.mplot3d import Axes3D
import kagglehub

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 10,
})
COLORS = plt.cm.tab10.colors

---
## 1 · Loading the data

**Session 1.** A dataset is a matrix $X \in \mathbb{R}^{n \times d}$ — $n$ samples (houses), $d$ features.

In [ ]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
df = pd.read_csv(os.path.join(path, 'train.csv'))
print(f'Loaded {df.shape[0]} houses × {df.shape[1]} columns')
print(f'SalePrice — mean ${df["SalePrice"].mean():,.0f}, median ${df["SalePrice"].median():,.0f}, std ${df["SalePrice"].std():,.0f}')
df.head(3)

---
## 2 · Exploratory Data Analysis

### 2.1 · Missing values

In [ ]:
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print(f'{len(miss)} columns have missing values')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(miss.index, miss.values / len(df) * 100, color='coral')
ax.set_xlabel('% missing'); ax.set_title('Missing values per feature')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

### 2.2 · Target distribution (Session 3 — distributions & normality)

`SalePrice` is right-skewed. The log transform makes it approximately Gaussian — important because MSE penalises large errors quadratically (Session 2), so a symmetric target distribution avoids disproportionate influence from expensive outliers.

In [ ]:
log_price = np.log(df['SalePrice'])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(df['SalePrice'], bins=50, edgecolor='white', alpha=0.8)
axes[0].set_title('SalePrice (raw)'); axes[0].set_xlabel('USD')
axes[1].hist(log_price, bins=50, color='orange', edgecolor='white', alpha=0.8)
axes[1].set_title('log(SalePrice)'); axes[1].set_xlabel('log(USD)')
sp_stats.probplot(log_price, plot=axes[2])
axes[2].set_title('Q-Q plot of log(SalePrice)')
axes[2].get_lines()[0].set(color=COLORS[0], markersize=3)
axes[2].get_lines()[1].set(color=COLORS[1])
plt.tight_layout(); plt.show()

_, p_shapiro = sp_stats.shapiro(log_price.sample(500, random_state=42))
print(f'Raw skewness: {df["SalePrice"].skew():.2f}  →  log skewness: {log_price.skew():.2f}')
print(f'Shapiro-Wilk p-value (log): {p_shapiro:.4f} — {"≈ normal" if p_shapiro > 0.05 else "not perfectly normal, but much improved"}')

### 2.3 · Feature selection & distributions

In [ ]:
features = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
    'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd',
    'YearBuilt', 'YearRemodAdd',
]
data = df[features + ['SalePrice']].dropna().copy()
data['logSalePrice'] = np.log(data['SalePrice'])
print(f'Working set: {data.shape[0]} houses × {len(features)} features (no NaNs)')

fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for ax, feat in zip(axes.ravel(), features):
    ax.hist(data[feat], bins=40, edgecolor='white', alpha=0.8)
    ax.set_title(feat, fontsize=9); ax.tick_params(labelsize=7)
plt.suptitle('Feature distributions', fontsize=12)
plt.tight_layout(); plt.show()

### 2.4 · Correlation heatmap

**Session 1.** Correlation is the normalised dot product of centred feature columns: $\rho_{ij} = \frac{\vec{x}_i \cdot \vec{x}_j}{\|\vec{x}_i\|\|\vec{x}_j\|}$.

In [ ]:
corr = data[features + ['logSalePrice']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, vmin=-1, vmax=1, linewidths=0.3)
ax.set_title('Pearson correlation (features + log price)')
plt.tight_layout(); plt.show()

print('Top correlations with logSalePrice:')
price_corr = corr['logSalePrice'].drop('logSalePrice').abs().sort_values(ascending=False)
for feat, r in price_corr.items():
    print(f'  {feat:>14s}  r = {corr.loc[feat, "logSalePrice"]:+.3f}')

### 2.5 · Scatter plots — top predictors vs price

In [ ]:
top6 = price_corr.index[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, feat in zip(axes.ravel(), top6):
    ax.scatter(data[feat], data['logSalePrice'], s=8, alpha=0.3)
    z = np.polyfit(data[feat], data['logSalePrice'], 1)
    xl = np.linspace(data[feat].min(), data[feat].max(), 100)
    ax.plot(xl, np.polyval(z, xl), color='red', lw=2, alpha=0.7)
    r = corr.loc[feat, 'logSalePrice']
    ax.set_xlabel(feat); ax.set_ylabel('log(SalePrice)')
    ax.set_title(f'{feat} (r = {r:+.2f})', fontsize=9)
plt.suptitle('Top 6 predictors vs target', fontsize=12)
plt.tight_layout(); plt.show()

### 2.6 · Outlier analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for ax, feat in zip(axes.ravel(), features):
    ax.boxplot(data[feat], vert=True)
    ax.set_title(feat, fontsize=9); ax.tick_params(labelsize=7)
    q1, q3 = data[feat].quantile(0.25), data[feat].quantile(0.75)
    iqr = q3 - q1
    n_out = ((data[feat] < q1 - 1.5*iqr) | (data[feat] > q3 + 1.5*iqr)).sum()
    ax.set_xlabel(f'{n_out} outliers', fontsize=7)
plt.suptitle('Boxplots — outlier detection (IQR method)', fontsize=12)
plt.tight_layout(); plt.show()

### 2.7 · Hypothesis test (Session 4 — statistical inference)

We test whether **OverallQual** rating significantly affects house prices:

- $H_0$: $\mu_{\text{high}} = \mu_{\text{low}}$ (no price difference between quality groups)
- $H_1$: $\mu_{\text{high}} \neq \mu_{\text{low}}$

We use **Welch's t-test** (unequal variances, Session 4).

In [ ]:
high_q = df[df['OverallQual'] >= 7]['SalePrice']
low_q  = df[df['OverallQual'] < 7]['SalePrice']
t_stat, p_val = sp_stats.ttest_ind(high_q, low_q, equal_var=False)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(low_q, bins=40, alpha=0.6, color=COLORS[1], edgecolor='white',
        label=f'Quality < 7 (n={len(low_q)}, mean=${low_q.mean():,.0f})')
ax.hist(high_q, bins=40, alpha=0.6, color=COLORS[0], edgecolor='white',
        label=f'Quality ≥ 7 (n={len(high_q)}, mean=${high_q.mean():,.0f})')
ax.set_xlabel('Sale Price ($)'); ax.set_ylabel('Frequency')
ax.set_title('SalePrice by Overall Quality Group'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Welch's t-test: t = {t_stat:.3f}, p = {p_val:.2e}")
print(f"Result: {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'} at α = 0.05")
print('OverallQual has a statistically significant effect on sale price.')

---
## 3 · Data preparation

### 3.1 · Standardisation

$$\tilde{x}_{ij} = \frac{x_{ij} - \bar{x}_j}{\sigma_j}$$

Standardising compresses the condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ of $X^\top X$, making the loss bowl more circular and widening the safe learning-rate window.

In [ ]:
X_raw = data[features].to_numpy(dtype=float)
y_all = np.log(data['SalePrice'].to_numpy())

mu, sigma = X_raw.mean(axis=0), X_raw.std(axis=0)
X_std = (X_raw - mu) / sigma

H_raw = (2 / len(X_raw)) * X_raw.T @ X_raw
H_std = (2 / len(X_std)) * X_std.T @ X_std
print(f'Condition number κ  BEFORE standardisation: {np.linalg.cond(H_raw):,.0f}')
print(f'Condition number κ  AFTER  standardisation: {np.linalg.cond(H_std):.1f}')
print(f'→ {np.linalg.cond(H_raw)/np.linalg.cond(H_std):,.0f}× improvement')

In [ ]:
X = np.hstack([np.ones((X_std.shape[0], 1)), X_std])
n, d = X.shape
print(f'Design matrix X: {n}×{d}  (bias + {d-1} features)')

idx = rng.permutation(n)
split = int(0.8 * n)
train_idx, test_idx = idx[:split], idx[split:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
print(f'Train {X_train.shape[0]} / Test {X_test.shape[0]}')

---
## 4 · Closed-form OLS — the benchmark (Session 1)

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

In [ ]:
def ols_closed_form(X, y):
    return np.linalg.inv(X.T @ X) @ X.T @ y

def mse(X, y, w):
    r = y - X @ w
    return float(r @ r) / len(y)

def r_squared(X, y, w):
    return 1 - np.sum((y - X @ w)**2) / np.sum((y - y.mean())**2)

t0 = time.time()
w_ols = ols_closed_form(X_train, y_train)
ols_time = time.time() - t0
ols_train_mse = mse(X_train, y_train, w_ols)
ols_test_mse  = mse(X_test, y_test, w_ols)

print('OLS weights:')
for name, val in zip(['bias'] + features, w_ols):
    print(f'  {name:>14s} : {val:+.4f}')
print(f'\nTrain — MSE: {ols_train_mse:.5f}  R²: {r_squared(X_train,y_train,w_ols):.4f}')
print(f'Test  — MSE: {ols_test_mse:.5f}  R²: {r_squared(X_test,y_test,w_ols):.4f}')
print(f'Time: {ols_time*1000:.2f} ms')

In [ ]:
feat_imp = pd.Series(np.abs(w_ols[1:]), index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(feat_imp.index, feat_imp.values, color='steelblue')
ax.set_xlabel('|weight| (standardised)'); ax.set_title('Feature importance from OLS')
plt.tight_layout(); plt.show()

---
## 5 · Gradient descent from scratch (Session 2)

### 5.1 · Loss, gradient, update rule

**MSE loss:** $\mathcal{L}(\vec{w}) = \frac{1}{n}\|y - X\vec{w}\|^2$

**Gradient** (chain rule → partial derivatives → vector form):
$$\nabla_{\vec{w}}\mathcal{L} = \frac{-2}{n} X^\top(y - X\vec{w})$$

**Update rule:** $\vec{w}_{t+1} = \vec{w}_t - \alpha\,\nabla\mathcal{L}(\vec{w}_t)$

In [ ]:
def grad_mse(X, y, w):
    return (-2.0 / len(y)) * X.T @ (y - X @ w)

def gradient_descent(X, y, lr, n_iters=1000, w0=None):
    n, d = X.shape
    w = np.zeros(d) if w0 is None else w0.copy()
    history = {'loss': [], 'grad_norm': [], 'weights': []}
    for _ in range(n_iters):
        g = grad_mse(X, y, w)
        w = w - lr * g
        history['loss'].append(mse(X, y, w))
        history['grad_norm'].append(np.linalg.norm(g))
        history['weights'].append(w.copy())
        if not np.isfinite(history['loss'][-1]):
            break
    return w, history

### 5.2 · Learning rate sweep

In [ ]:
lrs = [0.001, 0.01, 0.1, 0.3, 1.0]
histories = {}
w_finals  = {}
for lr in lrs:
    t0 = time.time()
    w, h = gradient_descent(X_train, y_train, lr=lr, n_iters=1000)
    dt = time.time() - t0
    histories[lr] = h; w_finals[lr] = w
    print(f'α={lr:<6}  final MSE = {h["loss"][-1]:.5f}  ({dt*1000:.1f} ms)')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, lr in enumerate(lrs):
    ax[0].plot(histories[lr]['loss'], label=f'$\\alpha={lr}$', color=COLORS[i])
    ax[1].semilogy(histories[lr]['loss'], label=f'$\\alpha={lr}$', color=COLORS[i])
for a in ax:
    a.axhline(ols_train_mse, color='k', ls='--', lw=1, label='OLS')
    a.set_xlabel('iteration'); a.legend(fontsize=7)
ax[0].set_ylabel('MSE'); ax[0].set_title('Loss curves (linear)')
ax[1].set_ylabel('MSE'); ax[1].set_title('Loss curves (log scale)')
plt.tight_layout(); plt.show()

### 5.3 · Gradient norm over iterations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, lr in enumerate(lrs):
    ax.semilogy(histories[lr]['grad_norm'], label=f'$\\alpha={lr}$', color=COLORS[i])
ax.set_xlabel('iteration'); ax.set_ylabel('$\\|\\nabla \\mathcal{L}\\|$')
ax.set_title('Gradient norm — convergence indicator'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### 5.4 · Hessian eigenvalue analysis (Session 1 × Session 2)

For MSE: $H = \frac{2}{n} X^\top X$. Its eigenvalues are real, $\geq 0$ (symmetric PSD). Stability bound:

$$\alpha < \frac{2}{\lambda_{\max}(H)}$$

In [ ]:
H = (2.0 / len(y_train)) * X_train.T @ X_train
eigvals_H = np.linalg.eigvalsh(H)
lam_max = eigvals_H[-1]; lam_min = eigvals_H[0]
alpha_max = 2.0 / lam_max
kappa = lam_max / max(lam_min, 1e-12)

print(f'λ_min = {lam_min:.4f},  λ_max = {lam_max:.4f}')
print(f'Safe learning rate: α < 2/λ_max = {alpha_max:.4f}')
print(f'Condition number κ = {kappa:.1f}')
print()
for lr in lrs:
    status = 'SAFE' if lr < alpha_max else 'DIVERGES'
    print(f'  α = {lr:<6} → {status}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, len(eigvals_H)+1), eigvals_H, color='teal')
ax.axhline(lam_max, ls='--', color='red', lw=1, label=f'$\\lambda_{{max}} = {lam_max:.2f}$')
ax.set_xlabel('index'); ax.set_ylabel('$\\lambda$')
ax.set_title('Hessian eigenvalue spectrum'); ax.legend()
plt.tight_layout(); plt.show()

### 5.5 · 3D loss surface

We slice through $w_0$ (bias) and $w_1$ (OverallQual), fixing the rest at OLS. Because MSE is convex quadratic, the surface is a **bowl** with a single global minimum.

In [ ]:
i0, i1 = 0, 1
w0_ols, w1_ols = w_ols[i0], w_ols[i1]
gp = 60; r = 0.7
w0g = np.linspace(w0_ols - r, w0_ols + r, gp)
w1g = np.linspace(w1_ols - r, w1_ols + r, gp)
W0, W1 = np.meshgrid(w0g, w1g)
LS = np.zeros_like(W0)
wb = w_ols.copy()
for ii in range(gp):
    for jj in range(gp):
        wt = wb.copy(); wt[i0] = W0[ii,jj]; wt[i1] = W1[ii,jj]
        LS[ii,jj] = mse(X_train, y_train, wt)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W0, W1, LS, cmap='viridis', alpha=0.85, edgecolor='none')
ax.scatter([w0_ols], [w1_ols], [ols_train_mse], color='red', s=60, zorder=5, label='OLS minimum')
ax.set_xlabel('$w_0$ (bias)'); ax.set_ylabel('$w_1$ (OverallQual)'); ax.set_zlabel('MSE')
ax.set_title('Loss surface — convex bowl'); ax.legend()
plt.tight_layout(); plt.show()

### 5.6 · Contour plot with GD trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
cs = ax.contour(W0, W1, LS, levels=30, cmap='viridis', alpha=0.6)
ax.clabel(cs, inline=True, fontsize=6)
ax.plot(w0_ols, w1_ols, 'r*', ms=14, zorder=5, label='OLS minimum')

safe_lrs = [lr for lr in lrs if lr < alpha_max]
for i, lr in enumerate(safe_lrs):
    traj = np.array(histories[lr]['weights'])
    ax.plot(traj[:, i0], traj[:, i1], 'o-', ms=1, lw=1, color=COLORS[i], label=f'$\\alpha={lr}$')
ax.set_xlabel('$w_0$ (bias)'); ax.set_ylabel('$w_1$ (OverallQual)')
ax.set_title('GD trajectories on loss contour'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### 5.7 · Weight convergence

In [ ]:
lr_demo = 0.1
traj = np.array(histories[lr_demo]['weights'])
fig, ax = plt.subplots(figsize=(9, 4))
for j, name in enumerate(['bias'] + features):
    ax.plot(traj[:, j], label=name, lw=1)
    ax.axhline(w_ols[j], color='gray', ls=':', lw=0.4)
ax.set_xlabel('iteration'); ax.set_ylabel('weight value')
ax.set_title(f'Weight convergence (α={lr_demo}), dashed = OLS target')
ax.legend(fontsize=6, ncol=3, loc='upper right')
plt.tight_layout(); plt.show()

---
## 6 · GD variants — Batch vs Mini-batch vs SGD (Session 2)

In [ ]:
def minibatch_gd(X, y, lr, n_epochs=80, batch_size=32, seed=0):
    rng_mb = np.random.default_rng(seed)
    n, d = X.shape; w = np.zeros(d); losses = []
    for _ in range(n_epochs):
        order = rng_mb.permutation(n)
        for start in range(0, n, batch_size):
            batch = order[start:start+batch_size]
            w = w - lr * grad_mse(X[batch], y[batch], w)
        losses.append(mse(X, y, w))
    return w, losses

lr_v = 0.05; n_ep = 80
_, l_batch = gradient_descent(X_train, y_train, lr=lr_v, n_iters=n_ep)
_, l_mini  = minibatch_gd(X_train, y_train, lr=lr_v, n_epochs=n_ep, batch_size=32)
_, l_sgd   = minibatch_gd(X_train, y_train, lr=lr_v, n_epochs=n_ep, batch_size=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for a in ax:
    a.plot(l_batch['loss'], label='Batch'); a.plot(l_mini, label='Mini-batch (b=32)'); a.plot(l_sgd, label='SGD (b=1)')
    a.axhline(ols_train_mse, color='k', ls='--', lw=1, label='OLS'); a.set_xlabel('epoch'); a.legend(fontsize=7)
ax[0].set_ylabel('MSE'); ax[0].set_title(f'GD variants (α={lr_v})')
ax[1].set_yscale('log'); ax[1].set_ylabel('MSE (log)'); ax[1].set_title('Log scale')
plt.tight_layout(); plt.show()

---
## 7 · GD vs closed-form — head-to-head

In [ ]:
alpha_safe = 0.9 * alpha_max
t0 = time.time()
w_gd, hist_gd = gradient_descent(X_train, y_train, lr=alpha_safe, n_iters=5000)
gd_time = time.time() - t0

print(f'{"":25s} {"OLS":>12s} {"GD (5k iters)":>14s}')
print('='*53)
print(f'{"Train MSE":25s} {mse(X_train,y_train,w_ols):>12.5f} {mse(X_train,y_train,w_gd):>14.5f}')
print(f'{"Test MSE":25s} {mse(X_test,y_test,w_ols):>12.5f} {mse(X_test,y_test,w_gd):>14.5f}')
print(f'{"Test R²":25s} {r_squared(X_test,y_test,w_ols):>12.4f} {r_squared(X_test,y_test,w_gd):>14.4f}')
print(f'{"Time (ms)":25s} {ols_time*1000:>12.2f} {gd_time*1000:>14.2f}')
print(f'{"||w_GD − w_OLS||":25s} {"—":>12s} {np.linalg.norm(w_gd-w_ols):>14.2e}')

In [ ]:
# Weight comparison bar chart.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

x_pos = np.arange(len(features))
bw = 0.35
axes[0].bar(x_pos - bw/2, w_ols[1:], bw, label='OLS', color=COLORS[0], alpha=0.8)
axes[0].bar(x_pos + bw/2, w_gd[1:], bw, label='GD', color=COLORS[1], alpha=0.8)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('weight'); axes[0].set_title('Learned weights'); axes[0].legend(fontsize=8)

y_pred = X_test @ w_gd
lo, hi = y_test.min(), y_test.max()
axes[1].scatter(y_test, y_pred, s=10, alpha=0.5)
axes[1].plot([lo,hi],[lo,hi],'k--',lw=1)
axes[1].set_xlabel('actual'); axes[1].set_ylabel('predicted'); axes[1].set_title('Predicted vs actual')

res = y_test - y_pred
axes[2].hist(res, bins=40, edgecolor='white', alpha=0.8, color='green')
axes[2].set_xlabel('residual'); axes[2].set_title(f'Residuals (σ={res.std():.3f})')
plt.tight_layout(); plt.show()

---
## 8 · Ridge regularisation (Session 5)

Ridge adds an $L_2$ penalty to prevent overfitting:

$$\mathcal{L}_{\text{Ridge}} = \frac{1}{n}\|y - X\vec{w}\|^2 + \frac{\lambda}{n}\|\vec{w}\|^2$$

The gradient becomes: $\nabla\mathcal{L} = \frac{-2}{n}X^\top(y - X\vec{w}) + \frac{2\lambda}{n}\vec{w}$

Larger $\lambda$ → more shrinkage toward zero → more bias, less variance.

In [ ]:
def ridge_gd(X, y, lr, lam, n_iters=1000):
    n, d = X.shape; w = np.zeros(d)
    losses = []
    for _ in range(n_iters):
        g = (-2.0/n) * X.T @ (y - X @ w) + (2.0*lam/n) * w
        w = w - lr * g
        r = y - X @ w
        losses.append(float(r @ r) / n + lam/n * float(w @ w))
    return w, losses

lambdas = [0, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_ws = {}
for lam in lambdas:
    w, _ = ridge_gd(X_train, y_train, lr=0.1, lam=lam, n_iters=1000)
    ridge_ws[lam] = w

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

W = np.array([ridge_ws[l][1:] for l in lambdas])
for j in range(W.shape[1]):
    axes[0].plot([str(l) for l in lambdas], W[:, j], 'o-', label=features[j], alpha=0.7, lw=1.5)
axes[0].axhline(0, color='k', ls='-', alpha=0.2)
axes[0].set_xlabel('λ'); axes[0].set_ylabel('weight')
axes[0].set_title('Weight shrinkage with increasing λ'); axes[0].legend(fontsize=6, ncol=2)

tr_r2 = [r_squared(X_train, y_train, ridge_ws[l]) for l in lambdas]
te_r2 = [r_squared(X_test,  y_test,  ridge_ws[l]) for l in lambdas]
axes[1].plot([str(l) for l in lambdas], tr_r2, 'o-', color=COLORS[0], lw=2, ms=8, label='Train R²')
axes[1].plot([str(l) for l in lambdas], te_r2, 's-', color=COLORS[1], lw=2, ms=8, label='Test R²')
axes[1].set_xlabel('λ'); axes[1].set_ylabel('R²')
axes[1].set_title('Ridge: bias-variance trade-off'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'{"λ":>8s} {"Train R²":>10s} {"Test R²":>10s} {"||w||":>10s}')
print('-'*42)
for lam in lambdas:
    w = ridge_ws[lam]
    print(f'{lam:>8.2f} {r_squared(X_train,y_train,w):>10.4f} {r_squared(X_test,y_test,w):>10.4f} {np.linalg.norm(w[1:]):>10.4f}')

---
## 9 · PCA — free exploration (Session 1)

**PCA recipe:** center → covariance matrix $\Sigma = \frac{1}{n-1}\tilde{X}^\top\tilde{X}$ → eigendecompose → sort → project.

In [ ]:
Sigma = (X_std.T @ X_std) / (len(X_std) - 1)
eigvals_S, eigvecs_S = np.linalg.eigh(Sigma)
order = np.argsort(eigvals_S)[::-1]
eigvals_S = eigvals_S[order]; eigvecs_S = eigvecs_S[:, order]

evr = eigvals_S / eigvals_S.sum()
cum = np.cumsum(evr)
n95 = int(np.argmax(cum >= 0.95) + 1)

print('Explained variance ratio:')
for i, (e, c) in enumerate(zip(evr, cum), 1):
    print(f'  PC{i}: {e:6.2%}   (cum {c:6.2%})')
print(f'\nComponents for ≥ 95% variance: {n95}/{len(features)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(range(1, len(evr)+1), evr, color='steelblue', label='individual')
axes[0].plot(range(1, len(evr)+1), cum, 'o-', color='orange', label='cumulative')
axes[0].axhline(0.95, color='red', ls='--', lw=1, label='95%')
axes[0].set_xlabel('component'); axes[0].set_ylabel('variance share')
axes[0].set_title('Scree plot'); axes[0].legend(fontsize=7)

Z_pca = X_std @ eigvecs_S[:, :2]
sc = axes[1].scatter(Z_pca[:, 0], Z_pca[:, 1], c=y_all, cmap='viridis', s=8, alpha=0.7)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA projection (colour = log price)')
plt.colorbar(sc, ax=axes[1], label='log(SalePrice)')

loadings = pd.DataFrame(eigvecs_S[:, :4], index=features, columns=[f'PC{i}' for i in range(1,5)])
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[2], linewidths=0.3)
axes[2].set_title('PC loadings')
plt.tight_layout(); plt.show()

### 9.1 · Regression performance vs number of PCs

In [ ]:
pca_results = {}
for k in range(1, len(features)+1):
    Z = X_std @ eigvecs_S[:, :k]
    Z = np.hstack([np.ones((Z.shape[0], 1)), Z])
    Ztr, Zte = Z[train_idx], Z[test_idx]
    w_pc = ols_closed_form(Ztr, y_train)
    pca_results[k] = {
        'r2_tr': r_squared(Ztr, y_train, w_pc),
        'r2_te': r_squared(Zte, y_test, w_pc),
    }

ks = list(pca_results.keys())
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, [pca_results[k]['r2_tr'] for k in ks], 'o-', color=COLORS[0], lw=2, ms=8, label='Train R²')
ax.plot(ks, [pca_results[k]['r2_te'] for k in ks], 's-', color=COLORS[1], lw=2, ms=8, label='Test R²')
ax.axvline(n95, color='gray', ls=':', alpha=0.5, label=f'{n95} PCs (95% var)')
ax.set_xlabel('Number of principal components'); ax.set_ylabel('R²')
ax.set_title('Regression performance vs PCA components'); ax.set_xticks(ks); ax.legend()
plt.tight_layout(); plt.show()

print(f'All {len(features)} features: Test R² = {r_squared(X_test,y_test,w_ols):.4f}')
print(f'Top {n95} PCs (95% var):  Test R² = {pca_results[n95]["r2_te"]:.4f}')

---
## 10 · Conclusions

**GD matches OLS** to machine precision ($\|\Delta w\| \approx 10^{-15}$) with the right learning rate.

**The Hessian eigenvalue bound** $\alpha < 2/\lambda_{\max}(H)$ predicted divergence before running — **linear algebra** (eigenvalues, S1) governs **calculus-based optimisation** (GD, S2).

**Standardisation** compressed $\kappa$ by orders of magnitude, making convergence practical.

**PCA** achieves dimensionality reduction with minimal R² loss.

**Ridge** ($L_2$) regularisation provides a natural extension of GD for complexity control.

| Session | Concept | Application |
|---|---|---|
| 1 — Linear Algebra | Eigenvalues, PCA, $X^\top X$ | OLS, Hessian spectrum, PCA decomposition |
| 2 — Calculus | Gradient, chain rule | GD implementation, learning rate bound |
| 3 — Probability | Distributions, normality | Log-transform, Q-Q plot, Shapiro-Wilk |
| 4 — Statistics | Hypothesis testing | Welch's t-test on quality groups |
| 5 — Advanced | Regularisation | Ridge GD from scratch |